In [1]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import memento

ModuleNotFoundError: No module named '__builtin__'

In [25]:
group_by = "perturbation"
reference = "control"

In [2]:
hallmark = gs.get_library('MSigDB_Hallmark_2020')

2026-05-06 09:13:39 | [INFO] Downloading and generating Enrichr library gene sets...


In [3]:
adata = ad.read_h5ad("../data/LSI_MimitouSmibert2021.hdf5") #

/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
adata = adata[adata.obs["perturbation"].dropna().index].copy()

In [5]:
sc.pp.log1p(adata)

In [6]:
adata.var.rename({"Unnamed: 0": "gene"}, axis=1, inplace=True)

In [7]:
adata.var.set_index("gene", inplace=True)

In [8]:
test_groups = adata.obs["perturbation"].unique()

In [9]:
rows = list()

In [10]:
name = 'MSigDB_Hallmark_2020'
geneset = hallmark
key = 'TNF-alpha Signaling via NF-kB'

subset = adata[:, np.isin(adata.var_names, geneset[key])].copy()

for test_group in test_groups:
    if test_group != "control":
        p, z, s = test(adata=subset, reference=reference, test_group=test_group, k="full", metric="sqeuclidean", rank=False, group_by=group_by)
        rows.append([f"{name} {key}", subset.shape[1], len(geneset[key]), test_group, p, z, s])

using CPU to calculate distance matrix.
creating distance graph with 2136 samples
counting cross matches.
using CPU to calculate distance matrix.
creating distance graph with 2280 samples
counting cross matches.
using CPU to calculate distance matrix.
creating distance graph with 2366 samples
counting cross matches.
using CPU to calculate distance matrix.
creating distance graph with 1734 samples
counting cross matches.
using CPU to calculate distance matrix.
creating distance graph with 1752 samples
counting cross matches.


In [11]:
results = pd.DataFrame(rows, columns=["Gene set", "#Genes used", "#Genes total", "Test group (knockout)", "P", "Z", "S"])
results["P_adj"] = multipletests(results["P"], method="fdr_bh")[1]

In [26]:
group_counts = subset.obs[group_by].value_counts()
min_count = group_counts.min()

print(f"Minimum group size: {min_count}", flush=True, file=sys.stderr)

sampled_indices = []
relative_support_dict = dict()

for g in group_counts.index:
    idx = np.where(adata.obs[group_by] == g)[0]
    sampled = np.random.choice(idx, min_count, replace=False)
    sampled_indices.append(sampled)
    relative_support_dict[g] = len(sampled) / len(idx)

sampled_indices = np.concatenate(sampled_indices)

Minimum group size: 621


In [27]:
subset_balanced = subset[sampled_indices].copy()

In [29]:
n_comps = min(50, subset.shape[1] - 1, subset.shape[0] - 1)
sc.pp.pca(subset_balanced, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(subset_balanced, groupby="perturbation", contrast="control")

Working... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:02:25

In [30]:
pd.options.display.float_format = "{:,.1e}".format
results.sort_values("Test group (knockout)")

,Gene set,#Genes used,#Genes total,Test group (knockout),P,Z,S,P_adj
3,MSigDB_Hallmark_2020 TNF-alpha Signaling via N...,195,200,CD3E,1.0e-38,-1.3e+01,1.0e+00,2.6e-38
4,MSigDB_Hallmark_2020 TNF-alpha Signaling via N...,195,200,CD3ECD4,9.0e-40,-1.3e+01,1.0e+00,4.5e-39
1,MSigDB_Hallmark_2020 TNF-alpha Signaling via N...,195,200,CD4,3.1e-01,-5.6e-01,1.0e+00,3.1e-01
0,MSigDB_Hallmark_2020 TNF-alpha Signaling via N...,195,200,NFKB2,6.5e-05,-3.9e+00,1.0e+00,8.1e-05
2,MSigDB_Hallmark_2020 TNF-alpha Signaling via N...,195,200,ZAP70,1.7e-18,-8.7e+00,1.0e+00,2.9e-18


In [31]:
tab.sort_index()

,distance,pvalue,significant,pvalue_adj,significant_adj
CD3E,3.9e-01,1.0e-04,True,6.0e-04,True
CD3ECD4,3.6e-01,1.0e-04,True,6.0e-04,True
CD4,2.1e-02,1.0e-04,True,6.0e-04,True
NFKB2,3.4e-02,1.0e-04,True,6.0e-04,True
ZAP70,2.1e-01,1.0e-04,True,6.0e-04,True
control,0.0e+00,1.0e+00,False,1.0e+00,False


In [32]:
subset

AnnData object with n_obs × n_vars = 5816 × 195
    obs: 'cell_barcode', 'BlacklistRatio', 'nDiFrags', 'nFrags', 'nMonoFrags', 'nMultiFrags', 'NucleosomeRatio', 'PromoterRatio', 'ReadsInBlacklist', 'ReadsInPromoter', 'ReadsInTSS', 'sample', 'TSSEnrichment', 'guide', 'sgCounts', 'sgTotal', 'sgSpec', 'perturbation', 'ReadsInPeaks', 'FRIP', 'disease', 'cancer', 'tissue_type', 'cell_line', 'organism', 'perturbation_type', 'nperts', 'percent_mito', 'percent_ribo', 'ncounts', 'ngenes', 'pseudo_bulk_100', 'pseudo_bulk_200', 'pseudo_bulk_500', 'split_50', 'split_40', 'split_30', 'split_20', 'split_10', 'split_1', 'XMatch_partner_NFKB2_vs_control', 'XMatch_partner_CD4_vs_control', 'XMatch_partner_ZAP70_vs_control', 'XMatch_partner_CD3E_vs_control', 'XMatch_partner_CD3ECD4_vs_control'
    var: 'ncounts', 'ncells'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [33]:
subset_balanced

AnnData object with n_obs × n_vars = 3726 × 195
    obs: 'cell_barcode', 'BlacklistRatio', 'nDiFrags', 'nFrags', 'nMonoFrags', 'nMultiFrags', 'NucleosomeRatio', 'PromoterRatio', 'ReadsInBlacklist', 'ReadsInPromoter', 'ReadsInTSS', 'sample', 'TSSEnrichment', 'guide', 'sgCounts', 'sgTotal', 'sgSpec', 'perturbation', 'ReadsInPeaks', 'FRIP', 'disease', 'cancer', 'tissue_type', 'cell_line', 'organism', 'perturbation_type', 'nperts', 'percent_mito', 'percent_ribo', 'ncounts', 'ngenes', 'pseudo_bulk_100', 'pseudo_bulk_200', 'pseudo_bulk_500', 'split_50', 'split_40', 'split_30', 'split_20', 'split_10', 'split_1', 'XMatch_partner_NFKB2_vs_control', 'XMatch_partner_CD4_vs_control', 'XMatch_partner_ZAP70_vs_control', 'XMatch_partner_CD3E_vs_control', 'XMatch_partner_CD3ECD4_vs_control'
    var: 'ncounts', 'ncells'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [37]:
gs.__version__

'1.2.1'